<a href="https://colab.research.google.com/github/asfiya-tehmeen/Clinical-Trial-Patient-Matching-Agent/blob/main/ehr_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 5 — EHR extraction agent

# Converts free-text clinical note → structured JSON profile

EXTRACTION_PROMPT = """
You are a clinical NLP specialist. Extract structured information from
this clinical note. Return ONLY valid JSON — no preamble, no markdown fences.

Extract exactly this schema:
{{
  "demographics": {{
    "age": <int>,
    "sex": "<M or F>",
    "location": "<city, state, country>"
  }},
  "primary_diagnosis": {{
    "name": "<full diagnosis name>",
    "icd10_guess": "<best ICD-10 code>",
    "stage": "<stage if available>",
    "histology": "<histology if available>"
  }},
  "biomarkers": [
    {{"name": "<biomarker>", "value": "<result>", "status": "<positive/negative/numeric>"}}
  ],
  "medications": [
    {{"name": "<drug name>", "dose": "<dose>", "frequency": "<frequency>"}}
  ],
  "labs": [
    {{"name": "<lab name>", "value": "<value>", "unit": "<unit>"}}
  ],
  "ecog_status": <int 0-4>,
  "prior_treatments": ["<treatment 1>", "<treatment 2>"],
  "comorbidities": ["<condition 1>", "<condition 2>"],
  "exclusion_flags": ["<anything that commonly excludes patients from trials>"],
  "key_search_terms": ["<3-6 best search terms for finding relevant trials>"]
}}

Clinical note:
\"\"\"
{ehr_text}
\"\"\"
"""

def call_grok(prompt: str, model: str, max_tokens: int = 1500) -> str:
    """Unified Grok API call — returns response text."""
    response = client.chat.completions.create(
        model      = model,
        max_tokens = max_tokens,
        messages   = [{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

def clean_json(raw: str) -> dict:
    """Strip markdown fences if model wraps output in them, then parse JSON."""
    if raw.startswith("```"):
        parts = raw.split("```")
        raw = parts[1] if len(parts) > 1 else raw
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

def extract_patient_profile(ehr_text: str) -> dict:
    print("🔍 Running EHR extraction with grok-3-mini...")
    raw = call_grok(EXTRACTION_PROMPT.format(ehr_text=ehr_text), "llama-3.1-8b-instant")
    profile = clean_json(raw)
    print("🔍 Running EHR extraction with llama-3.1-8b-instant (Groq)...")
    return profile

patient_profile = extract_patient_profile(PATIENT_EHR)
print("\n📋 Structured patient profile:")
print(json.dumps(patient_profile, indent=2))



🔍 Running EHR extraction with grok-3-mini...
🔍 Running EHR extraction with llama-3.1-8b-instant (Groq)...

📋 Structured patient profile:
{
  "demographics": {
    "age": 61,
    "sex": "F",
    "location": "Boston, MA, USA"
  },
  "primary_diagnosis": {
    "name": "Stage IIIB non-small cell lung cancer",
    "icd10_guess": "C34.92",
    "stage": "IIIB",
    "histology": "adenocarcinoma"
  },
  "biomarkers": [
    {
      "name": "EGFR exon 19 deletion",
      "value": "positive",
      "status": "positive"
    },
    {
      "name": "ALK",
      "value": "negative",
      "status": "negative"
    },
    {
      "name": "PD-L1 TPS",
      "value": "45%",
      "status": "numeric"
    }
  ],
  "medications": [
    {
      "name": "metformin",
      "dose": "1000mg",
      "frequency": "BID"
    },
    {
      "name": "lisinopril",
      "dose": "10mg",
      "frequency": "QD"
    },
    {
      "name": "ondansetron",
      "dose": "PRN",
      "frequency": "PRN"
    }
  ],
  "labs": [
 